In [ ]:
import os

import MDAnalysis as md

import shaker

shaker.set_options(show=False, verbose=True, progress=False)

main_dir = os.getcwd()

# AGI Molecule Parameterization with `Shaker`

This notebook follows a simple workflow for AGI parameterization of a molecule using `Shaker`.

AGI assumes that a mapping and reference simulation is already provided in the cell below.

## 1. Mapping & SASA sanity check.
Maps the AA reference simulation and runs initial SASA calculations (CG-Mapped vs AA) to sanity check the mapping.

Mapping and reference AA sims should be provided by the user.

In [ ]:
author = "Luis Borges-Araujo"

itp_header = [
    f"; Parameterised by {author} and our friendly neighbourhood AI overlords @ ENS de Lyon and the cloud somewhere, 2026.\n",
    "; Molecular name: WAR \n",
    "; SMILES: C(F)(F)(F)OC1C=CC(C(N2CCC(C3C4N=CC(OCCCC)=CC=4N=CN=3)CC2)=O)=CC=1 \n",
]

## AA reference sims (already PBC treated and removed solvent.)
GRO = "../tutorials/AA_references/BasicParam/AA.pdb"
XTC = "../tutorials/AA_references/BasicParam/AA.xtc"
resname = "MOL"

## User supplied mapping in the shaker format.
mapping = {
    ## resname
    "MOL": {
        ## bead name; type(opt.);   charge(opt.);         Mapping.
        "R1": {
            "type": "SX3",
            "charge": 0,
            "atoms": ["Cl1", "Cl1", "C0B", "C0B", "C0A", "C05"],
        },
        "R2": {
            "type": "SX3",
            "charge": 0,
            "atoms": ["C06", "C06", "C05", "C08", "Cl0", "Cl0"],
        },
        "R3": {
            "type": "TC5",
            "charge": 0,
            "atoms": ["C09", "C09", "C08", "H0V", "H0W", "H0W", "C0A", "H0X"],
        },
        "P1": {"type": "SP2", "charge": 0, "atoms": ["O04", "C03", "N02", "H0U"]},
        "R4": {
            "type": "TC5",
            "charge": 0,
            "atoms": ["C00", "C0D", "C0D", "C0E", "H0Y", "H0Y"],
        },
        "R5": {"type": "TN6a", "charge": 0, "atoms": ["C0E", "N0F", "N0F", "C0G"]},
        "R6": {
            "type": "TC5",
            "charge": 0,
            "atoms": ["C01", "C01", "H0T", "H0T", "C00", "C0G"],
        },
        "P2": {"type": "SP2", "charge": 0, "atoms": ["N0H", "H0S", "C0I", "O0J"]},
        "C1": {
            "type": "TC1",
            "charge": 0,
            "atoms": ["H0Q", "H0R", "C0K", "C0M", "C0M", "H0N", "H0O", "H0P"],
        },
    },
}

#########
## Map ##
#########
shaker.map_aa2cg(GRO, XTC, mapping)
print("\n\n")
shaker.render_2dMapping(GRO, resname, mapping, show_bead_type=True)

##########
## SASA ##
##########
shaker.run_SASA(
    "CG_Mapped",
    f"{main_dir}/cg_mapped.gro",
    f"{main_dir}/cg_mapped.xtc",
    resname,
    isCG=True,
    mapping=mapping,
)
shaker.run_SASA("AA", GRO, XTC, resname)
fig, ax, sasa_values = shaker.plot_sasa_dir()

## 2. Estimate initial bonded terms

Start minimal and build up over iterations: **bonds/constraints first**, then angles, then dihedrals,
adding terms only where the comparison with the reference shows they're needed.

- `shaker.bead_neighbours` lists which beads are bonded (from the AA structure) and any bead rings.
  Neighbour pairs are candidate bonds; paths of 3 and 4 beads are candidate angles and dihedrals.
  Beads joined through an unmapped atom all count as neighbours, so extra triangles are expected.
- `shaker.bonded_estimator` fits harmonic parameters to the mapped AA distributions and returns an itp
  block. Very stiff bonds become constraints. Treat the values as starting guesses.

In [ ]:
neighbours = shaker.bead_neighbours(GRO, mapping, resname)

In [ ]:
## bonded terms to estimate. You must fill this.
estimator_dist_tgts = [
    # ['A1', 'B2'],    # Example
]

estimator_ang_tgts = [
    # ['A1', 'B2', 'B3'],    # Example
]

estimator_harm_dihed_tgts = [
    # ['A1', 'B2', 'B3', 'B4'],  # Example
]

estimator_imp_dihed_tgts = [
    # ['A1', 'B2', 'B3', 'B4'],  # Example
]

u_aa = md.Universe(f"{main_dir}/cg_mapped.gro", f"{main_dir}/cg_mapped.xtc")
footer = shaker.bonded_estimator(
    u_aa,
    resname=resname,
    dist_tgts=estimator_dist_tgts,
    ang_tgts=estimator_ang_tgts,
    harm_dihed_tgts=estimator_harm_dihed_tgts,
    imp_dihed_tgts=estimator_imp_dihed_tgts,
    use_indices=True,
)
print(footer)

In [ ]:
## Defaults: no virtual sites. The commented block below overwrites these when enabled.
itp_mapping, vs_lines = mapping, []

## OPTIONAL: rigid/fused units (> 3 beads in one rigid unit) as a constrained frame + virtual sites.
## Uncomment and set the bead names. Run after map_aa2cg, before writing the itp.
# import copy
# os.chdir(main_dir)                                  # align_* writes into the cwd
# unit  = ["B1", "B2", "B3", "B4", "B5"]              # all beads of the rigid unit
# frame = ["B1", "B2", "B4", "B5"]                    # 3-4 frame beads (constrained); the rest become virtual sites
# shaker.align_mol_to_single_traj(f"{main_dir}/cg_mapped.gro", f"{main_dir}/cg_mapped.xtc",
#                                 selection=f"resname {resname}",
#                                 align_selection="name " + " ".join(unit))
# vs_lines, vs_masses = shaker.generate_virtual_sitesN(      # or generate_virtual_sites3 for a 3-bead frame
#     md.Universe(f"{main_dir}/average_molecule.gro"), frame,
#     selection="name " + " ".join(unit), mass_split="equal",
#     mapping={resname: {b: mapping[resname][b] for b in unit}},  # ONLY the unit's beads, or every other bead gets mass 0
#     resname=resname)
# itp_mapping = copy.deepcopy(mapping)                # masses go in a copy; the mapping itself stays untouched
# for b, d in vs_masses[resname].items():
#     itp_mapping[resname][b]["mass"] = d["mass"]
# print("\n".join(vs_lines))

## 3. Prepare and run a simple CG system

Simulate one copy of the molecule in water, to compare its bonded distributions against the AA reference.

- `shaker.write_initial_CGitp` writes `initial_CG.itp` once (iteration 1) from the mapping plus the estimated terms. Afterwards edit the itp by hand; delete it to regenerate.
- `shaker.prepare_setup_water` builds the solvated box (run once). `shaker.runSim` runs minimisation,
  relaxation and a 200 ns production run (~10 min on a GPU) with Shaker's bundled Martini 3 files.
- Each `runSim` call deletes the previous run's files in `CG_WAT/` and checkpoints the new trajectory and
  itp in `CG_WAT/iter_<N>_<timestamp>/`. `shaker.list_iterations("CG_WAT")` lists them;
  `keep_last=N` prunes older ones.

In [ ]:
dir_run = f"{main_dir}/CG_WAT"
os.makedirs(dir_run, exist_ok=True)
os.chdir(dir_run)
try:
    ## Written once (iteration 1). Afterwards, edit CG_WAT/initial_CG.itp by hand.
    if not os.path.exists("initial_CG.itp"):
        shaker.write_initial_CGitp(
            resname, itp_mapping, header=itp_header, footer=[footer, *vs_lines]
        )
finally:
    os.chdir(main_dir)

In [ ]:
## Set up the initial system.
## Simple box of water with the molecule of interest (built once; topol.top includes the itp by path).
os.chdir(dir_run)
try:
    if not os.path.exists("topol.top"):
        shaker.prepare_setup_water("../cg_mapped.gro", n_mols=1)
    shaker.runSim()
finally:
    os.chdir(main_dir)

## 4. Analyse each run
Run the targets cell (once per change to the terms), set `note`, then run the cell below. It compares AA, this run and the previous run, recomputes SASA and the overlap matrix, and records the iteration.

- **OC** (overlap coefficient, 0–1): ✓ ≥ 0.80, ⚠ ≥ 0.65, ✗ below. **W** (Wasserstein): 0 = identical.
- **mean-off**: CG mean differs from AA by more than one AA sd.
- Track terms you don't parameterize as well (end-to-end distances, unparameterized dihedrals): a molecule can match every fitted term and still have the wrong shape.
- SASA should stay within 5 % of AA; the overlap matrix shows which beads carry the error.
- Iterations are checkpointed in `CG_WAT/iter_*`; the table printed at the end shows whether each change helped.

In [ ]:
## bonded terms to probe. You must fill this.
dist_tgts = [
    # ['A1', 'B2'],    # Example
]

ang_tgts = [
    # ['A1', 'B2', 'B3'],    # Example
]

dihed_tgts = [
    # ['A1', 'B2', 'B3', 'B4'],  # Example
]

In [ ]:
## Analyse the latest run: bonded (AA vs CG vs previous), SASA, overlap, record.

note = "Write your iteration note here! :D"  # note to label this iteration with

############
## Bonded ##
############
os.chdir(main_dir)
u_aa = md.Universe(f"{main_dir}/cg_mapped.gro", f"{main_dir}/cg_mapped.xtc")
u_cg = md.Universe(f"{main_dir}/CG_WAT/pbc.pdb", f"{main_dir}/CG_WAT/pbc.xtc")
AA_bonded = shaker.measure_bonded_terms(u_aa, resname, dist_tgts, ang_tgts, dihed_tgts)
CG_bonded = shaker.measure_bonded_terms(u_cg, resname, dist_tgts, ang_tgts, dihed_tgts)
bonded, labels = [AA_bonded, CG_bonded], ["AA", "CG"]
iters = shaker.list_iterations(f"{main_dir}/CG_WAT")
if len(iters) >= 2:
    u_prev = md.Universe(str(iters[-2] / "pbc.pdb"), str(iters[-2] / "pbc.xtc"))
    bonded.append(
        shaker.measure_bonded_terms(u_prev, resname, dist_tgts, ang_tgts, dihed_tgts)
    )
    labels.append("Prev. CG")
_ = shaker.plot_bonded_distributions(*bonded, labels=labels, outfile="cleanbonds")

##########
## SASA ##
##########
print("\n\n\n")
shaker.run_SASA(
    "CG_Simulated",
    f"{main_dir}/CG_WAT/pbc.pdb",
    f"{main_dir}/CG_WAT/pbc.xtc",
    resname,
    isCG=True,
    mapping=mapping,
)
_, _, sasa_values = shaker.plot_sasa_dir()

#############
## Overlap ##
#############
print("\n\n\n")
oc, _, _ = shaker.assess_overlap_matrix(u_aa, u_cg, resname=resname)

######################
## Record iteration ##
######################
print("\n\n\n")
_ = shaker.record_iteration(
    f"{main_dir}/CG_WAT",
    note=note,
    overlap=oc,
    bonded=(AA_bonded, CG_bonded),
    sasa=sasa_values,
)

## 5. Iterate until the model is good enough

Each iteration: change the bonded terms (directly in `CG_WAT/initial_CG.itp`, with a comment on each edited
line), re-run section 3, then the analysis cell. Add terms in stages: angles first, then dihedrals.

Stop when the run is clean (no LINCS warnings), SASA is within 5 % of AA, single-peaked terms are ✓/⚠
and not mean-off, and the mean overlap gain falls below 0.01. Then write the results cell below:
history table, final itp, and remaining caveats.

## 6. Results
**Final model:** iteration N, `CG_WAT/initial_CG.itp` (checkpoint: `CG_WAT/iter_<N-1>_<timestamp>/`)
**Stopped because:** <all stop criteria met / 10-iteration limit / other>

| iter | change | SASA % of AA | mean overlap | bonded ✓ / ⚠ / ✗ | mean-off |
|---|---|---|---|---|---|
| 1 | bonds/constraints only | | | | |
| 2 | | | | | |
| … | | | | | |

**Run stability:** <LINCS warnings / crashes across runs, and any runs that failed and why>

**Key design choices**
- <e.g. rigid units: frame + virtual sites, which beads>
- <terms that are not obvious, and why they were chosen>
- <terms deliberately left out, and why>

**Remaining issues** (each: term, AA vs CG, why it isn't fixed)
- <multimodal terms matched on mean/sd only>
- <dihedrals not allowed (near-linear angle) and what covers them instead>
- <handedness: one-sided AA torsions fitted as sampled>

**Notes for the next user**
- Hand edits in the itp: iterations <N…M>, each line commented.
- Cells not to re-run: <list>
- Anything run outside the notebook: <list, or "none">